In [1]:
# Imports

import ray
from ray.rllib.algorithms.ppo import PPOConfig

import IPython.core.display_functions

from src.parsers import HMParser, CotevParser
from src.algorithms.rl import EnergyCommunitySequentialV12

from src.utils import load_multiple_upacs_pv, iterate_resources, create_ppo_policies
from src.priorities import EmpiricalPriority

import warnings
warnings.filterwarnings('ignore')

2025-03-04 02:05:34,525	WARNING deprecation.py:50 -- DeprecationWarning: `DirectStepOptimizer` has been deprecated. This will raise an error in the future!
2025-03-04 02:05:34,776	WARNING deprecation.py:50 -- DeprecationWarning: `build_tf_policy` has been deprecated. This will raise an error in the future!
2025-03-04 02:05:34,778	WARNING deprecation.py:50 -- DeprecationWarning: `build_policy_class` has been deprecated. This will raise an error in the future!


In [2]:
# Data parsing

# EC data for non-renewable generators and batteries
data_ec = HMParser(file_path='/Users/ecgomes/DataspellProjects/pyecom/data/EC_V4.xlsx', ec_id=1)
data_ec.parse()

# EV data from the EV4EU simulator
data_ev = CotevParser(population_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/population_731.csv',
                      driving_history_path=
                      '/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/ev_driving_history_731.csv',
                      assigned_segments_path='/Users/ecgomes/DataspellProjects/pyecom/data/simulation_dataframes_2years/assigned_segments_731.csv',
                      parse_date_start='2019',
                      parse_date_end='2020')
data_ev.parse()

# UPAC Data load
data_upacs = load_multiple_upacs_pv('/Users/ecgomes/Documents/PhD/UPAC data/upac*_pv.csv', resample='H')

In [3]:
# Create resources for the training environment

dataset_resources = iterate_resources(u=data_upacs, c=data_ec, e=data_ev, mode='monthly')

In [4]:
# Get the execution order

execution_order = EmpiricalPriority(dataset_resources['2019-01'])
execution_order = execution_order.calculate_priority()
execution_order

,priority,type,name
6,5.594149,<class 'src.resources.load.Load'>,load_06
8,5.447441,<class 'src.resources.load.Load'>,load_09
7,5.375024,<class 'src.resources.load.Load'>,load_02
9,5.321487,<class 'src.resources.load.Load'>,load_13
5,5.000000,<class 'src.resources.load.Load'>,load_08
2,4.609039,<class 'src.resources.generator.Generator'>,ren_generator_02
4,4.604803,<class 'src.resources.generator.Generator'>,ren_generator_13
1,4.448629,<class 'src.resources.generator.Generator'>,ren_generator_06
0,4.114321,<class 'src.resources.generator.Generator'>,ren_generator_08
3,4.000000,<class 'src.resources.generator.Generator'>,ren_generator_09


In [5]:
# The order we will use are the strings in the execution order, except the loads

order = [x for x in execution_order['name'] if 'load' not in x and 'ren_gen' not in x]
order

['ev_02',
 'ev_04',
 'ev_01',
 'ev_05',
 'ev_03',
 'storage_03',
 'storage_01',
 'storage_02',
 'aggregator']

In [6]:
# Create the environment and check if everything is ok

temp_env = EnergyCommunitySequentialV12(ren_generators=dataset_resources[list(dataset_resources.keys())[0]][:5],
                                        generators=[],
                                        loads=dataset_resources[list(dataset_resources.keys())[0]][5:10],
                                        storages=dataset_resources[list(dataset_resources.keys())[0]][10:13],
                                        evs=dataset_resources[list(dataset_resources.keys())[0]][13:-1],
                                        aggregator=dataset_resources[list(dataset_resources.keys())[0]][-1],
                                        storage_penalty=1,
                                        ev_penalty=1,
                                        balance_penalty=1,
                                        execution_order=order)
temp_env.reset()
terminations = truncations = {a: False for a in temp_env.agents}
terminations['__all__'] = False
truncations['__all__'] = False
while not terminations['__all__'] and not truncations['__all__']:

    actions = temp_env.action_space_sample()
    next_obs, rewards, terminations, truncations, infos = temp_env.step(actions)

print('Terminated: {}'.format(terminations['__all__']))

Terminated: True


In [7]:
# Create the policies to train

# The keys of the dictionary respect the class names of the agents
gammas = {'Generator': 0.0, 'Storage': 0.9, 'Vehicle': 0.9, 'Aggregator': 0.9}

# Create the policies, one for each agent. Each policy has the name of the agent.
policies = create_ppo_policies(temp_env, gammas)

2025-03-04 02:06:10,011	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-03-04 02:06:10,012	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module_api=True`. When RLModule API are enabled, exploration_config can not be set. If you want to implement custom exploration behaviour, please modify the `forward_exploration` method of the RLModule at hand. On configs that have a default exploration config, this must be done with `config.exploration_config={}`.
2025-03-04 02:06:10,012	WARNING algorithm_config.py:2578 -- Setting `exploration_config={}` because you set `_enable_rl_module

In [8]:
# Create individual networks for each agent

model_cfgs = {}
for agent in temp_env.agents:

    if agent.startswith('storage'):
        model_cfg = {'use_lstm': True,
                     'lstm_cell_size': 128,
                     'fcnet_hiddens': [32, 32],
                     'fcnet_activation': 'relu',
                     'lstm_use_prev_action': True,
                     'lstm_use_prev_reward': True,
                     'vf_share_layers': False}

    elif agent.startswith('ev'):
        model_cfg = {'use_lstm': True,
                     'lstm_cell_size': 128,
                     'fcnet_hiddens': [32, 32],
                     'fcnet_activation': 'relu',
                     'lstm_use_prev_action': True,
                     'lstm_use_prev_reward': True,
                     'vf_share_layers': False}

    else:
        model_cfg = {'fcnet_hiddens': [1],}

    model_cfgs[agent] = model_cfg

In [9]:
# Create an RLlib Algorithm instance from a PPOConfig to learn how to
# act in the above environment.

from ray.tune import register_env
from ray import tune, train
from ray.tune.schedulers import AsyncHyperBandScheduler
from ray.tune.stopper import CombinedStopper, MaximumIterationStopper, TrialPlateauStopper

ray.shutdown()
ray.init()

IMPORT_PENALTY = 1 #100
EXPORT_PENALTY = 1 #10
STORAGE_ACTION_PENALTY = 50 #100
STORAGE_ACTION_REWARD = 5 #10
EV_ACTION_PENALTY = 1 #1000
EV_ACTION_REWARD = 5 #10
EV_REQUIREMENT_PENALTY = 50 #3000
BALANCE_PENALTY = 5000 #20000

MAX_ITER = 500

checkpoint = None
checkpoint_path = None
algo = None
current_best = None

# Build a loop for using separate resources on a daily basis

temp_resources = dataset_resources['2019-01']

env = EnergyCommunitySequentialV12(ren_generators=temp_resources[:5],
                                   generators=[],
                                   loads=temp_resources[5:10],
                                   storages=temp_resources[10:13],
                                   evs=temp_resources[13:-1],
                                   aggregator=temp_resources[-1],
                                   storage_penalty=STORAGE_ACTION_PENALTY,
                                   ev_penalty=EV_REQUIREMENT_PENALTY,
                                   balance_penalty=BALANCE_PENALTY,
                                   execution_order=order)
register_env("EC_Seq_V2", lambda config: env)

# Define the PPOConfig
_config = (PPOConfig()
           .environment(env="EC_Seq_V2", disable_env_checking=False)
           .training(train_batch_size=256,
                     lr=5e-5, #tune.grid_search([0.001, 0.0001]),
                     gamma=0.99,
                     use_gae=True,
                     use_critic=True,
                     use_kl_loss=True,
                     clip_param=0.1,
                     grad_clip=4,
                     )
           .exploration(exploration_config={})
           .framework('torch')
           .multi_agent(policies=policies,
                        policies_to_train=list(policies.keys())[:-1],
                        policy_mapping_fn=(lambda agent_id, episode, worker, **kwargs:
                                           agent_id),
                        algorithm_config_overrides_per_module=model_cfgs)
           .rollouts(batch_mode='complete_episodes', #'complete_episodes',
                     num_rollout_workers=10,
                     rollout_fragment_length='auto'))

# Clear the Jupyter cell output
IPython.core.display_functions.clear_output()


stopper = CombinedStopper(
    MaximumIterationStopper(max_iter=MAX_ITER),
    TrialPlateauStopper(metric='episode_reward_mean', mode='max', metric_threshold=0.01)
)

scheduler = AsyncHyperBandScheduler(time_attr="training_iteration",
                                    max_t=MAX_ITER,
                                    metric="episode_reward_mean",
                                    mode="max",
                                    grace_period=10)

# Train the algorithm with Tuner
tuner = tune.Tuner(
    "PPO",
    param_space=_config,
    run_config=train.RunConfig(stop={'training_iteration': MAX_ITER, 'episode_reward_mean': -50.0}),
    tune_config=tune.TuneConfig(scheduler=scheduler, num_samples=1)#, metric="episode_reward_mean", mode="max"),
)

results = tuner.fit()

2025-03-04 02:08:01,469	INFO tune.py:1143 -- Total run time: 107.72 seconds (97.67 seconds for the tuning loop).
2025-03-04 02:08:01,470	WARNING tune.py:1158 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: Tuner.restore(path="/Users/ecgomes/ray_results/PPO_2025-03-04_02-06-13", trainable=...)


In [11]:
 results.get_best_result('episode_reward_mean', 'max').get_best_checkpoint('episode_reward_mean', 'max').path

'/Users/ecgomes/ray_results/PPO_2025-02-03_21-27-06/PPO_EC_Seq_V1_9e473_00000_0_2025-02-03_21-27-06/checkpoint_000000'